# Machine Sound Classification: Subset Downloader (High-Speed Compression)

This notebook downloads or subsets samples from each class and compresses them using **pigz**.

## 1. Setup & Configuration
Run this cell to initialize everything.

In [ ]:
try:
    import kagglehub
except ImportError:
    !pip install -q kagglehub
    import kagglehub

import os
import shutil
import subprocess
import time
import random
from pathlib import Path
from tqdm.auto import tqdm

# --- CONFIGURATION ---
DATASET_HANDLE = "karimmyasser/machine-large-dataset"
SAMPLES_PER_CLASS = 1000  # Change this to 2500 if needed

IS_KAGGLE = Path("/kaggle/input").exists()
if IS_KAGGLE:
    # Intermediate storage in /tmp
    TMP_DIR = Path(f"/tmp/subset_{SAMPLES_PER_CLASS}")
    # Final zip in /kaggle/working
    OUTPUT_FILE = Path(f"/kaggle/working/machine_subset_{SAMPLES_PER_CLASS}.tar.gz")
    dataset_path = Path("/kaggle/input/datasets/karimmyasser/machine-large-dataset")
    if not dataset_path.exists():
        dataset_path = Path("/kaggle/input/machine-large-dataset")
else:
    TMP_DIR = Path(f"data/subset_{SAMPLES_PER_CLASS}")
    OUTPUT_FILE = Path(f"machine_subset_{SAMPLES_PER_CLASS}.tar.gz")
    dataset_path = Path(kagglehub.dataset_download(DATASET_HANDLE))

MACHINES = ["Machine 1", "Machine 2", "Machine 3"]
CLASSES = ["Normal", "Abnormal"]

print(f"Target: {SAMPLES_PER_CLASS} samples per class")
print(f"Temp Directory: {TMP_DIR}")
print(f"Output File: {OUTPUT_FILE}")

## 2. Create Subset
Copies the files into the temp directory.

In [ ]:
if TMP_DIR.exists():
    shutil.rmtree(TMP_DIR)
TMP_DIR.mkdir(parents=True, exist_ok=True)

for machine in MACHINES:
    for cls in CLASSES:
        src_dir = dataset_path / "data" / machine / "machine_data" / cls
        if not src_dir.exists():
            print(f"Warning: Source directory not found: {src_dir}")
            continue
            
        wav_files = list(src_dir.glob("*.wav"))
        print(f"Found {len(wav_files)} samples for {machine} - {cls}")
        
        count = min(len(wav_files), SAMPLES_PER_CLASS)
        subset_files = random.sample(wav_files, count)
            
        dst_dir = TMP_DIR / machine / cls
        dst_dir.mkdir(parents=True, exist_ok=True)
        
        for f in tqdm(subset_files, desc=f"Copying {machine}/{cls}"):
            shutil.copy2(f, dst_dir / f.name)

print(f"\nSubset creation complete! Files stored in: {TMP_DIR}")

## 3. High-Speed Compression
Uses pigz to compress the temp directory.

In [ ]:
import subprocess
import time
from pathlib import Path

# Ensure variables are available even if cells were run out of order
if 'TMP_DIR' not in locals():
    print("Error: TMP_DIR not defined. Please run the Setup cell first.")
elif 'OUTPUT_FILE' not in locals():
    print("Error: OUTPUT_FILE not defined. Please run the Setup cell first.")
else:
    print(f"Compressing {TMP_DIR} using pigz...")
    start_time = time.time()

    parent_dir = TMP_DIR.parent
    folder_name = TMP_DIR.name

    # Check if pigz is available
    has_pigz = subprocess.run(["which", "pigz"], capture_output=True).returncode == 0

    if has_pigz:
        print("Using parallel compression (pigz)")
        cmd = f"tar -cf - -C {parent_dir} {folder_name} | pigz > {OUTPUT_FILE}"
    else:
        print("pigz not found, falling back to standard tar/gzip")
        cmd = f"tar -czf {OUTPUT_FILE} -C {parent_dir} {folder_name}"

    process = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    if process.returncode == 0:
        elapsed = time.time() - start_time
        print(f"\nSuccess! File created at: {OUTPUT_FILE}")
        print(f"Compression took: {elapsed:.2f} seconds")
    else:
        print("Compression failed!")
        print(process.stderr)